# LSTM Sentiment Classification: Positive or Negative




## 1. Import libraries

In [45]:
import time

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 2. Create the dataset

In [46]:
df = pd.read_csv('./household_power_consumption.txt', sep=';', na_values='?')
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index(df['DateTime'], inplace=True)
df.drop(columns=['Date', 'Time', 'DateTime'], inplace=True)
print(df.head())

                     Global_active_power  Global_reactive_power  Voltage  \
DateTime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388                  0.502   233.74   
2006-12-16 17:28:00                3.666                  0.528   235.68   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
DateTime                                                                
2006-12-16 17:24:00              18.4             0.0             1.0   
2006-12-16 17:25:00              23.0             0.0             1.0   
2006-12-16 17:26:00              23.0             0.0             2.0   
2006-12-16 17:27:00              23.0             0.0             1.0   
2006-12-16 17:28:00          

## 3. Missing Data and Data Smoothing

In [ ]:
df.interpolate(method='time', inplace=True)

# 1. Resample to hourly aggregates
df_hourly = df.resample('h').mean()
df_hourly['Global_active_power'] = np.log1p(df_hourly['Global_active_power'])
# df_hourly = df_hourly.rolling(window=24, min_periods=1).mean()


print(f"Remaining null values: {df_hourly.isna().sum().sum()}")

Remaining null values: 0


## 4. Train/Test split 


In [48]:
test_cutoff = df_hourly.index.max() - pd.DateOffset(months=6)

train_val_df = df_hourly.loc[df_hourly.index < test_cutoff]
test_df      = df_hourly.loc[df_hourly.index >= test_cutoff]

# Chronological validation split (last 15% of remaining data)
val_len  = int(len(train_val_df) * 0.15)
train_df = train_val_df.iloc[:-val_len]
val_df   = train_val_df.iloc[-val_len:]

# Fit scaler ONLY on train data, transform all splits
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[['Global_active_power']])
val_scaled   = scaler.transform(val_df[['Global_active_power']])
test_scaled  = scaler.transform(test_df[['Global_active_power']])

def make_sliding_windows(data, input_hours=168, output_hours=24):
    X, y = [], []
    for i in range(len(data) - input_hours - output_hours + 1):
        X.append(data[i : i + input_hours])
        y.append(data[i + input_hours : i + input_hours + output_hours])
    return np.array(X), np.array(y)

X_train, y_train = make_sliding_windows(train_scaled)
X_val, y_val     = make_sliding_windows(val_scaled)
X_test, y_test   = make_sliding_windows(test_scaled)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.squeeze(-1), dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

## 6. LSTM model


In [49]:
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, output_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out

model = SimpleLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(25):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(batch_X), batch_y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1} | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t.to(device)).cpu().numpy()

y_test_actual = np.expm1(scaler.inverse_transform(y_test.squeeze(-1)))
y_pred_actual = np.expm1(scaler.inverse_transform(y_pred_scaled))
epsilon = 1e-5
mape = np.mean(np.abs((y_test_actual - y_pred_actual) / np.maximum(y_test_actual, epsilon))) * 100
print(f"\nTest MAPE: {mape:.2f}%")

Epoch 1 | Loss: 0.0340
Epoch 2 | Loss: 0.0227
Epoch 3 | Loss: 0.0224
Epoch 4 | Loss: 0.0239
Epoch 5 | Loss: 0.0268
Epoch 6 | Loss: 0.0260
Epoch 7 | Loss: 0.0254
Epoch 8 | Loss: 0.0245
Epoch 9 | Loss: 0.0291
Epoch 10 | Loss: 0.0234
Epoch 11 | Loss: 0.0222
Epoch 12 | Loss: 0.0256
Epoch 13 | Loss: 0.0228
Epoch 14 | Loss: 0.0181
Epoch 15 | Loss: 0.0229
Epoch 16 | Loss: 0.0198
Epoch 17 | Loss: 0.0226
Epoch 18 | Loss: 0.0249
Epoch 19 | Loss: 0.0194
Epoch 20 | Loss: 0.0198
Epoch 21 | Loss: 0.0201
Epoch 22 | Loss: 0.0188
Epoch 23 | Loss: 0.0173
Epoch 24 | Loss: 0.0165
Epoch 25 | Loss: 0.0179

Test MAPE: 55.85%


## 7. Inference Latency

In [50]:
model.eval()
sample = X_test_t[:1].to(device)   # one household, past 7 days
with torch.no_grad():
    for _ in range(10):            # warm-up
        model(sample)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        model(sample)
    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - start) / 100 * 1000

print(f"Latency: {latency_ms:.2f} ms")

Latency: 0.61 ms


## 8. Anomaly Alerting

In [51]:
# Build a contiguous hourly residual series from non-overlapping 24h forecasts
resid = (y_test_actual - y_pred_actual)[::24].flatten()
sigma = resid.std()
threshold = 3 * sigma
exceed = np.abs(resid) > threshold

# Flag only runs of 2+ consecutive exceedances
alert = np.zeros(len(resid), dtype=bool)
run = 0
for i, e in enumerate(exceed):
    run = run + 1 if e else 0
    if run >= 2:
        alert[i] = alert[i - 1] = True

false_alarm_rate = alert.mean() * 100
print(f"Alerts: {int(alert.sum())} / {len(alert)} hours flagged")
print(f"False-alarm rate: {false_alarm_rate:.2f}%")

Alerts: 32 / 4248 hours flagged
False-alarm rate: 0.75%
